# Apply Frame Classifier

This notebook applies the trained frame classifier to all ADHD/Autism mention contexts and writes a reusable frame-label handoff for downstream LSC notebooks.

In [1]:
from __future__ import annotations

import hashlib
import json
import pickle
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "01_classification":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

CONTEXT_PATH = PROJECT_ROOT / "data/interim/lsc/contexts/lsc_mention_contexts.parquet"
TARGET_POOL_PATH = PROJECT_ROOT / "data/interim/lsc/classification/frame_target_context_pool.csv"
MODEL_DIR = PROJECT_ROOT / "data/processed/lsc/classification"
MODEL_PATH = MODEL_DIR / "hierarchical_frame_logistic_models.pkl"
METADATA_PATH = MODEL_DIR / "frame_classifier_metadata.json"
OUTPUT_PATH = MODEL_DIR / "lsc_target_context_frame_labels.csv"
SUMMARY_PATH = MODEL_DIR / "lsc_frame_counts_by_year_unit.csv"


## Load Model and Current Target Contexts

The target-context pool is refreshed from the shared context table before prediction so downstream handoffs cannot drift when the corpus is rebuilt.


In [2]:
if not MODEL_PATH.exists():
    print(f"No trained model found yet: {MODEL_PATH.relative_to(PROJECT_ROOT)}")
    raise SystemExit("Train and validate the classifier first.")

with MODEL_PATH.open("rb") as handle:
    models = pickle.load(handle)
metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))

pool_columns = [
    "doc_id",
    "url",
    "registered_domain",
    "analysis_unit",
    "target_group",
    "raw_form",
    "matched_text",
    "mention_start_char",
    "mention_end_char",
    "target_sentence_plus_adjacent",
    "lsc_year",
    "source_year",
]
contexts = pd.read_parquet(CONTEXT_PATH, columns=pool_columns)
target_contexts = contexts.loc[contexts["analysis_unit"].isin(["ADHD", "Autism"])].copy()


def stable_context_id(row: pd.Series) -> str:
    value = "|".join(
        str(row.get(column, ""))
        for column in ["doc_id", "analysis_unit", "raw_form", "mention_start_char", "mention_end_char"]
    )
    return hashlib.sha1(value.encode("utf-8")).hexdigest()[:16]


target_contexts = target_contexts.dropna(subset=["target_sentence_plus_adjacent", "lsc_year"])
target_contexts = target_contexts.loc[target_contexts["target_sentence_plus_adjacent"].str.strip().ne("")].copy()
target_contexts["context_id"] = target_contexts.apply(stable_context_id, axis=1)
target_contexts["year_band"] = pd.cut(
    target_contexts["lsc_year"].astype(int),
    bins=[2013, 2017, 2022, 2026],
    labels=["early_2014_2017", "mid_2018_2022", "late_2023_2026"],
)

duplicate_context_ids = target_contexts["context_id"].duplicated().sum()
if duplicate_context_ids:
    raise ValueError(f"Context ID collision or duplicate mention rows found: {duplicate_context_ids}")

target_contexts = target_contexts.sort_values(["analysis_unit", "lsc_year", "context_id"]).reset_index(drop=True)
TARGET_POOL_PATH.parent.mkdir(parents=True, exist_ok=True)
target_contexts.to_csv(TARGET_POOL_PATH, index=False)
print(f"Target contexts to label: {len(target_contexts):,}")
print(f"Refreshed target-context pool: {TARGET_POOL_PATH.relative_to(PROJECT_ROOT)}")


Target contexts to label: 96,864
Refreshed target-context pool: data/interim/lsc/classification/frame_target_context_pool.csv


## Embed and Predict

In [3]:
try:
    from sentence_transformers import SentenceTransformer
except ImportError as error:
    raise ImportError("Install the msc-nlp environment with sentence-transformers before applying the classifier.") from error


def classifier_text(frame: pd.DataFrame) -> list[str]:
    return [
        f"TARGET={row.analysis_unit}\nPASSAGE={row.target_sentence_plus_adjacent}"
        for row in frame.itertuples(index=False)
    ]


def derive_frame_from_predictions(substantive: bool, clinical: bool | pd.NA, lived: bool | pd.NA) -> str:
    if not substantive:
        return "non_substantive_or_insufficient"
    clinical_bool = bool(clinical)
    lived_bool = bool(lived)
    if clinical_bool and lived_bool:
        return "mixed"
    if clinical_bool:
        return "clinical_only"
    if lived_bool:
        return "lived_only"
    return "substantive_other"

embedder = SentenceTransformer(metadata["embedding_model"])
x_all = embedder.encode(classifier_text(target_contexts), normalize_embeddings=True, show_progress_bar=True)

labels = target_contexts.copy().reset_index(drop=True)
labels["p_substantive"] = models["substantive_target_discourse"].predict_proba(x_all)[:, 1]
labels["p_clinical_given_substantive"] = models["clinical_frame_present"].predict_proba(x_all)[:, 1]
labels["p_lived_given_substantive"] = models["lived_experience_frame_present"].predict_proba(x_all)[:, 1]

labels["predicted_substantive_target_discourse"] = labels["p_substantive"].ge(0.5).astype("boolean")
labels["predicted_clinical_frame_present"] = labels["p_clinical_given_substantive"].ge(0.5).astype("boolean")
labels["predicted_lived_experience_frame_present"] = labels["p_lived_given_substantive"].ge(0.5).astype("boolean")
labels.loc[~labels["predicted_substantive_target_discourse"], "predicted_clinical_frame_present"] = pd.NA
labels.loc[~labels["predicted_substantive_target_discourse"], "predicted_lived_experience_frame_present"] = pd.NA

labels["predicted_derived_frame"] = [
    derive_frame_from_predictions(substantive, clinical, lived)
    for substantive, clinical, lived in zip(
        labels["predicted_substantive_target_discourse"],
        labels["predicted_clinical_frame_present"],
        labels["predicted_lived_experience_frame_present"],
    )
]

labels["w_non_substantive_or_insufficient"] = 1 - labels["p_substantive"]
labels["w_clinical_only"] = labels["p_substantive"] * labels["p_clinical_given_substantive"] * (1 - labels["p_lived_given_substantive"])
labels["w_lived_only"] = labels["p_substantive"] * (1 - labels["p_clinical_given_substantive"]) * labels["p_lived_given_substantive"]
labels["w_mixed"] = labels["p_substantive"] * labels["p_clinical_given_substantive"] * labels["p_lived_given_substantive"]
labels["w_substantive_other"] = labels["p_substantive"] * (1 - labels["p_clinical_given_substantive"]) * (1 - labels["p_lived_given_substantive"])
labels["classifier_version"] = metadata["classifier_version"]
labels["codebook_version"] = metadata["codebook_version"]
labels["prompt_version"] = metadata["prompt_version"]


/opt/anaconda3/envs/msc-nlp/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Batches:   0%|          | 0/3027 [00:00<?, ?it/s]

Batches:   0%|          | 1/3027 [00:07<6:26:04,  7.66s/it]

Batches:   0%|          | 2/3027 [00:08<3:06:57,  3.71s/it]

Batches:   0%|          | 3/3027 [00:09<2:03:12,  2.44s/it]

...

Batches: 100%|█████████▉| 3021/3027 [14:43<00:00, 11.03it/s]

Batches: 100%|█████████▉| 3023/3027 [14:44<00:00, 11.01it/s]

Batches: 100%|█████████▉| 3025/3027 [14:44<00:00,  9.20it/s]

Batches: 100%|██████████| 3027/3027 [14:44<00:00,  9.92it/s]

Batches: 100%|██████████| 3027/3027 [14:44<00:00,  3.42it/s]

## Save Frame Labels

Before saving, the notebook checks that the refreshed target pool and processed frame labels contain exactly the same context IDs.


In [4]:
pool_ids = set(pd.read_csv(TARGET_POOL_PATH, usecols=["context_id"])["context_id"])
label_ids = set(labels["context_id"])
if pool_ids != label_ids:
    raise ValueError("Refreshed target pool and frame-label outputs do not contain the same context IDs.")

labels.to_csv(OUTPUT_PATH, index=False)
summary = labels.groupby(["lsc_year", "analysis_unit", "predicted_derived_frame"], as_index=False).size().rename(columns={"size": "contexts"})
summary.to_csv(SUMMARY_PATH, index=False)

print("Wrote frame-label outputs:")
for path in [TARGET_POOL_PATH, OUTPUT_PATH, SUMMARY_PATH]:
    print(f"- {path.relative_to(PROJECT_ROOT)}")

summary.head()


Wrote frame-label outputs:
- data/interim/lsc/classification/frame_target_context_pool.csv
- data/processed/lsc/classification/lsc_target_context_frame_labels.csv
- data/processed/lsc/classification/lsc_frame_counts_by_year_unit.csv


,lsc_year,analysis_unit,predicted_derived_frame,contexts
0,2014,ADHD,clinical_only,1286
1,2014,ADHD,lived_only,310
2,2014,ADHD,mixed,183
3,2014,ADHD,non_substantive_or_insufficient,1284
4,2014,ADHD,substantive_other,55
